In [ ]:
!pip install ultralytics pycocotools pandas -q
import os, json, shutil, pandas as pd
from sklearn.model_selection import train_test_split
import yaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive; drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile, os

# Unzip
with zipfile.ZipFile("/content/archive (1).zip", 'r') as zip_ref:
    zip_ref.extractall("/content/cocoblur")
print("Blurry images extracted to /content/cocoblur/")

with zipfile.ZipFile("/content/annotations_trainval2017.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/annotations")
print("Annotations extracted to /content/annotations/")

blurry_root = "/content/cocoblur/val2017_blurred_deterministic"
annotations_path = "/content/annotations/annotations/instances_val2017.json"

print(f"Blurry: {blurry_root}")
print(f"Annotations: {annotations_path}")
print(f"Images found: {len(os.listdir(blurry_root))}")

Blurry images extracted to /content/cocoblur/
Annotations extracted to /content/annotations/
Blurry: /content/cocoblur/val2017_blurred_deterministic
Annotations: /content/annotations/annotations/instances_val2017.json
Images found: 5001


In [ ]:
import numpy as np
import json

In [ ]:
import json, os
from collections import defaultdict

blurry_root = "/content/cocoblur/val2017_blurred_deterministic"

# Step 1: Create image metadata from blurry files
blurry_files = sorted(os.listdir(blurry_root))
new_img_id = {filename: i for i, filename in enumerate(blurry_files)}
images = [{'id': i, 'file_name': f, 'height': 480, 'width': 640} for i, f in enumerate(blurry_files)]

print(f"Created {len(images)} image records")

# Step 2: Load ORIGINAL COCO annotations (train2017)
with open(annotations_path) as f:
    coco = json.load(f)

# Step 3: Map COCO filenames to new IDs (handle filename mismatches)
coco_filename_to_id = {img['file_name']: img['id'] for img in coco['images']}
blurry_filename_to_new_id = new_img_id

val_annotations = []
matched = 0

for ann in coco['annotations']:
    # Find matching blurry filename
    for coco_fname, coco_old_id in coco_filename_to_id.items():
        if ann['image_id'] == coco_old_id:
            # Strip
            blurry_fname = coco_fname.replace('_blurred.jpg', '.jpg').replace('_blurry.jpg', '.jpg')

            if blurry_fname in blurry_filename_to_new_id:
                ann['image_id'] = blurry_filename_to_new_id[blurry_fname]
                val_annotations.append(ann)
                matched += 1
                break

print(f"Matched {matched} annotations to {len(blurry_files)} blurry images")
print(f"New annotations: {len(val_annotations)}")

# Save corrected dataset
corrected_coco = {
    'images': images[:1000],
    'annotations': val_annotations,
    'categories': coco['categories']
}

with open('/content/cocoblur_task4.json', 'w') as f:
    json.dump(corrected_coco, f)


Created 5001 image records
Matched 36781 annotations to 5001 blurry images
New annotations: 36781


In [ ]:
# Setting up folders
import os, shutil, json
from sklearn.model_selection import train_test_split

base_dir = "/content/task4_cocoblur"
for split in ['train', 'val', 'test']:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)


In [ ]:
# Load annotations
with open('/content/cocoblur_task4.json') as f:
    dataset = json.load(f)

print(f"Dataset: {len(dataset['images'])} images, {len(dataset['annotations'])} annotations")

Dataset: 1000 images, 36781 annotations


In [ ]:
img_ids = [img['id'] for img in dataset['images']]
np.random.seed(42)
sampled_ids = np.random.choice(img_ids, 200, replace=False)

train_ids, temp_ids = train_test_split(sampled_ids, test_size=0.4, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

splits = {
    'train': train_ids,
    'val': val_ids,
    'test': test_ids
}

print("Split sizes:", {k: len(v) for k,v in splits.items()})

Split sizes: {'train': 120, 'val': 40, 'test': 40}


In [ ]:
# Convert to YOLO format
import pandas as pd

def coco_to_yolo_bbox(bbox, img_width=640, img_height=480):
    x, y, w, h = bbox
    return [
        (x + w/2) / img_width,
        (y + h/2) / img_height,
        w / img_width,
        h / img_height
    ]

total_boxes = 0
stats = []

for split_name, img_ids in splits.items():
    split_img_dir = f"{base_dir}/images/{split_name}"
    split_label_dir = f"{base_dir}/labels/{split_name}"

    split_boxes = 0
    for img_id in img_ids:
        img_info = next(img for img in dataset['images'] if img['id'] == img_id)
        blurry_path = f"{blurry_root}/{img_info['file_name']}"
        label_path = f"{split_label_dir}/{img_info['file_name'].replace('.jpg', '.txt')}"


        shutil.copy(blurry_path, f"{split_img_dir}/{img_info['file_name']}")

        boxes = []
        for ann in dataset['annotations']:
            if ann['image_id'] == img_id:
                yolo_box = coco_to_yolo_bbox(ann['bbox'])
                boxes.append(f"{ann['category_id']-1} {' '.join(f'{coord:.6f}' for coord in yolo_box)}")

        with open(label_path, 'w') as f:
            f.write('\n'.join(boxes))

        split_boxes += len(boxes)

    stats.append({'split': split_name, 'images': len(img_ids), 'boxes': split_boxes})
    print(f"{split_name}: {len(img_ids)} images, {split_boxes} boxes")

pd.DataFrame(stats)

train: 120 images, 804 boxes
val: 40 images, 287 boxes
test: 40 images, 228 boxes


,split,images,boxes
0,train,120,804
1,val,40,287
2,test,40,228


In [ ]:
#create dataset
!pip install PyYAML -q
import yaml

dataset_yaml = {
    'path': base_dir,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 80,
    'names': [c['name'] for c in dataset['categories']]
}

with open('/content/cocoblur_task4.yaml', 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)


In [ ]:
# Train the model

from ultralytics import YOLO

model = YOLO('yolov8n.pt')

# Train blurry COCO
results = model.train(
    data='/content/cocoblur_task4.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device="cpu",
    project='runs/task4_blurry_coco',
    name='v1_blurry_coco',
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,
    optimizer='AdamW',
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    seed=42,
    save_period=10
)

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cocoblur_task4.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v1_blurry_coco, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, pe